> 在线讲义：[第 41 章 · 数据交接、顺序与状态管理：什么时候能读、能覆盖、能释放](https://wikieden.github.io/ai-infra-from-constraints/lessons/41-handoff-state.html) · 本书《从约束推导 AI Infra》

# 第 41 章 · 数据交接、顺序与状态管理：什么时候能读、能覆盖、能释放

配套讲义：仓库里 `ai-infra/notes/lessons/41-handoff-state.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



共享传输状态、OpenURMA 的状态尺寸与缓存溢出点、例题 7.4、槽位释放与背压取自 AII 第 7 章 §7.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB = 2**10, 2**20

**函数 `state_bytes`**：groups=None：每关系独占；groups=K：K 组共享传输状态。

In [ ]:
def state_bytes(L: int, P: int, e: int = 256, r: int = 64, t: int = 1024, groups: int | None = None) -> float:
    """groups=None：每关系独占；groups=K：K 组共享传输状态。"""
    if groups is None:
        return L * e + L * P * (r + t)
    return L * e + L * P * r + groups * P * t

**函数 `impl_state`**：小工具。

In [ ]:
def impl_state(N: int, M: int) -> tuple[int, int]:
    return 512 * N * M + 32 * N, 52 * N + 56 * M

**函数 `overflow_n`**：小工具。

In [ ]:
def overflow_n(cache: int = 256 * KiB) -> tuple[int, int]:
    pair = next(n for n in range(1, 10**5) if 512 * n * n + 32 * n > cache)
    ub = next(n for n in range(1, 10**5) if 108 * n > cache)
    return pair, ub

**函数 `ordering`**：(串行时 C 完成, 只保留 A→B 时 C 完成, 发布链完成)。

In [ ]:
def ordering(write: float = 20, recover: float = 80, notify: float = 2, c: float = 10) -> tuple[float, float, float]:
    """(串行时 C 完成, 只保留 A→B 时 C 完成, 发布链完成)。"""
    chain = write + recover + notify
    return chain + c, c, chain

**函数 `slots_sim`**：每 1 μs 最多提交一次，提交后 lat 完成；软件每 period 处理 batch 条完成通知并释放槽位。返回 (全部传完, 全部槽位释放)。

In [ ]:
def slots_sim(n_ops: int = 16, slots: int = 8, lat: float = 5, batch: int = 4, period: float = 20) -> tuple[float, float]:
    """每 1 μs 最多提交一次，提交后 lat 完成；软件每 period 处理 batch 条完成通知并释放槽位。返回 (全部传完, 全部槽位释放)。"""
    t, free, submit, done = 0.0, slots, [], []
    released = 0
    release_times = []
    k = 1
    while len(submit) < n_ops:
        while free == 0:
            # 等下一次批处理
            ready = sorted(s + lat for s in submit)[released:]
            tick = period * k
            k += 1
            n = len([x for x in ready if x <= tick][:batch])
            released += n
            free += n
            release_times += [tick] * n
            t = max(t, tick)
        submit.append(t)
        free -= 1
        t += 1
    done = [s + lat for s in submit]
    while released < n_ops:
        ready = sorted(done)[released:]
        tick = period * k
        k += 1
        n = len([x for x in ready if x <= tick][:batch])
        released += n
        release_times += [tick] * n
    return max(done), max(release_times)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：64 个端点 × 128 个目标")
p, s = state_bytes(64, 128), state_bytes(64, 128, groups=1)
print(f"  独占 {p / MiB:.2f} MiB，共享 {s / MiB:.2f} MiB（约 1/{p / s:.0f}）")
for K in (3, 4, 8):
    print(f"  {K} 组隔离：{state_bytes(64, 128, groups=K) / KiB:.0f} KiB")
a, b = impl_state(1024, 1024)
print(f"  实现尺寸 N = M = 1024：逐对 {a / MiB:.0f} MiB，UB {b / KiB:.0f} KiB，{a / b:.0f} 倍；256 KiB 缓存溢出点 {overflow_n()}")

In [ ]:
print("\n第二步：例题 7.4")
for rec in (80, 200):
    cs, ci, ch = ordering(recover=rec)
    print(f"  恢复 {rec} μs：串行 C 在 {cs:.0f} μs，独立 C 在 {ci:.0f} μs，发布链 {ch:.0f} μs")

In [ ]:
print("\n第三步：槽位与完成通知")
for n in (8, 16):
    d, r = slots_sim(slots=n)
    print(f"  {n} 个槽位：全部传完 {d:.0f} μs，槽位全部释放 {r:.0f} μs")
rate = 4 / 20e-6
print(f"  完成通知 {rate:,.0f} 项/s × 8 KiB = {rate * 8192 / 1e9:.2f} GB/s；50 GB/s 要 {50e9 / 8192 / 1e6:.1f} M 项/s")

## 3. 代码里才看得到的

1. `slots_sim` 每批只处理已经传完的那几条，按完成先后取前 4 条，所以 8 槽时第 9 次提交要等 20 μs 的第一批。
2. `ordering` 的串行安排里 C 等的是整条发布链（写 + 恢复 + 通知）。
3. `impl_state` 的 UB 一项没有 $NM$ 项：关系绑定放在软件映射里，网卡不存。

## 4. 自己改一改

- 给 `slots_sim` 加"每处理一条通知要 1 μs"的模型，求能维持的最大提交速率。
- 把 `ordering` 改成三条独立链，算全组和每条链的完成时刻。
- 用 `overflow_n` 算 1 MiB 缓存的溢出点。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 托盘送到、对方看到、对方用完，是三个时刻；
> 工号各管各的，同城包裹共一条线；
> 柜子先放再发短信，别家的货不用等；先读的旧值，晚点返回也还是旧的；
> 桌子多容高峰，收桌快才接得多。